In [1]:
!pip install kagglehub

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install dspy

In [3]:
import importlib
import os
import ssl
import sys
import certifi

# Fix Windows / conda SSL context issues by forcing certifi as the trust bundle.
# `ssl.create_default_context()` can fail with ASN1 / certificate store errors when importing dspy.
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# Clear stale partial imports from previous failed attempts.
for module_name in list(sys.modules):
    if module_name in {'dspy', 'litellm', 'aiohttp', 'httpx', 'yarl', 'multidict'} \
       or module_name.startswith(('dspy.', 'litellm.', 'aiohttp.', 'httpx.', 'yarl.', 'multidict.')):
        del sys.modules[module_name]

importlib.invalidate_caches()

_orig_ssl_create_default_context = ssl.create_default_context

def _fixed_create_default_context(*args, **kwargs):
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    ctx.load_verify_locations(cafile=certifi.where())
    return ctx

ssl.create_default_context = _fixed_create_default_context

try:
    import dspy
    import pandas as pd
    import numpy as np
    import pathlib as Pathlib
finally:
    ssl.create_default_context = _orig_ssl_create_default_context

In [12]:
!pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/914.9 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 3.2 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 4.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 4.3 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]



In [4]:
import os

import kagglehub
from kagglehub.auth import get_kaggle_credentials, set_kaggle_api_token

api_token = "KGAT_41bbe755704d283bc4d612cb30c93ef9"
os.environ['KAGGLE_API_TOKEN'] = api_token
set_kaggle_api_token(api_token)

creds = get_kaggle_credentials()
print('Kaggle credentials set:', bool(creds), creds)

# Download latest version
path = kagglehub.competition_download('kaggle-llm-science-exam')

print("Path to competition files:", path)

Kaggle credentials set.
Kaggle credentials set: True KaggleApiCredentials(username=None, key=None, api_key='KGAT_41bbe755704d283bc4d612cb30c93ef9')


100%|██████████| 72.5k/72.5k [00:00<00:00, 179kB/s]

Extracting files...
Path to competition files: C:\Users\devis\.cache\kagglehub\competitions\kaggle-llm-science-exam


In [7]:
data_path = Pathlib.Path('\kaggle\input\competitions\kaggle-llm-science-exam')
train = pd.read_csv(data_path/'train.csv', index_col='id')
train.head()


FileNotFoundError: [Errno 2] No such file or directory: '\\kaggle\\input\\competitions\\kaggle-llm-science-exam\\train.csv'

In [25]:
test = pd.read_csv(data_path / 'test.csv', index_col='id')
test.head()

,prompt,A,B,C,D,E
id,,,,,,
0,Which of the following statements accurately d...,MOND is a theory that reduces the observed mis...,MOND is a theory that increases the discrepanc...,MOND is a theory that explains the missing bar...,MOND is a theory that reduces the discrepancy ...,MOND is a theory that eliminates the observed ...
1,Which of the following is an accurate definiti...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...
2,Which of the following statements accurately d...,The triskeles symbol was reconstructed as a fe...,The triskeles symbol is a representation of th...,The triskeles symbol is a representation of a ...,The triskeles symbol represents three interloc...,The triskeles symbol is a representation of th...
3,What is the significance of regularization in ...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...
4,Which of the following statements accurately d...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...


In [26]:
local_model=dspy.LM('groq/llama-3.3-70b-versatile', api_key='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')
dspy.configure(lm=local_model)

In [30]:
class MultpleChoice(dspy.Signature):
    """Answer a multiple-choice science question by ranking the options."""
    
    prompt = dspy.InputField(desc="Questions have mutiple choice answers.")
    option_a = dspy.InputField(desc="Option A")
    option_b = dspy.InputField(desc="Option B")
    option_c = dspy.InputField(desc="Option C")
    option_d = dspy.InputField(desc="Option D")
    option_e = dspy.InputField(desc="Option E")
    
    top3_ranking = dspy.OutputField(
        desc="The top 3 most likely correct options ranked best to worst, "
             "as a comma-separated list of letters, e.g. 'A,C,E'"
    )

In [31]:
# Build the predictor module
class TopKPredictor(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predict = dspy.ChainOfThought(MultpleChoice)
    
    def forward(self, prompt, A, B, C, D, E):
        result = self.predict(
            prompt=prompt,
            option_a=A,
            option_b=B,
            option_c=C,
            option_d=D,
            option_e=E
        )
        return result

In [32]:
# Instantiate
predictor = TopKPredictor()

In [51]:
train.iloc[0]

prompt    Which of the following statements accurately d...
A         MOND is a theory that reduces the observed mis...
B         MOND is a theory that increases the discrepanc...
C         MOND is a theory that explains the missing bar...
D         MOND is a theory that reduces the discrepancy ...
E         MOND is a theory that eliminates the observed ...
answer                                                    D
Name: 0, dtype: object

In [52]:
test_row = {
    "prompt"    :train.iloc[0]['prompt'],
    "option_a"  :train.iloc[0]['A'],
    "option_b"  :train.iloc[0]['B'],
    "option_c"  :train.iloc[0]['C'],
    "option_d"  :train.iloc[0]['D'],
    "option_e"  :train.iloc[0]['E']
}

In [53]:
test_row

{'prompt': 'Which of the following statements accurately describes the impact of Modified Newtonian Dynamics (MOND) on the observed "missing baryonic mass" discrepancy in galaxy clusters?',
 'option_a': 'MOND is a theory that reduces the observed missing baryonic mass in galaxy clusters by postulating the existence of a new form of matter called "fuzzy dark matter."',
 'option_b': 'MOND is a theory that increases the discrepancy between the observed missing baryonic mass in galaxy clusters and the measured velocity dispersions from a factor of around 10 to a factor of about 20.',
 'option_c': 'MOND is a theory that explains the missing baryonic mass in galaxy clusters that was previously considered dark matter by demonstrating that the mass is in the form of neutrinos and axions.',
 'option_d': 'MOND is a theory that reduces the discrepancy between the observed missing baryonic mass in galaxy clusters and the measured velocity dispersions from a factor of around 10 to a factor of about

In [59]:
result = predictor(
    prompt=test_row["prompt"],
    A=test_row["option_a"], B=test_row["option_b"], C=test_row["option_c"], D=test_row["option_d"], E=test_row["option_e"]
)

print(result.top3_ranking)
#print(result.rationale)

D, E, A


AttributeError: 'Prediction' object has no attribute 'rationale'

In [58]:
dspy.history()

AttributeError: module 'dspy' has no attribute 'history'